In [ ]:
# Cell 1: Install dependencies
!pip install -q scanpy anndata igraph leidenalg scikit-learn scipy requests
!pip install -q cellxgene-census

In [ ]:
# Cell 2: Mount Drive and upload project files
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/sigreg_ablation/'
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

# Upload all .py files to /content/ before running cells below:
# cell_jepa.py, cell_sigreg.py, losses.py, preprocessing.py,
# trainer.py, metrics.py, compare_pbmc3k.py, run_ablation.py
print('Drive mounted. Results will be saved to:', RESULTS_DIR)

In [ ]:
# Cell 3: Smoke test — both variants, 200 cells, 1 epoch (~5 min on CPU)
# Run this first to confirm everything loads correctly before the full run.
import subprocess, os

REPO_DIR = '/content'
result = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_ablation.py'),
     '--variant', 'all',
     '--smoke_test',
     '--device', 'cpu',
     '--results_file', 'results_ablation_smoke.txt'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# Cell 4: Full ablation — both variants on PBMC-3K (~60 min on A100)
# Requires: smoke test (Cell 3) passed without errors.
import subprocess, time, os, threading

REPO_DIR = '/content'

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_ablation.py'),
     '--variant', 'all',
     '--device', 'cuda',
     '--pretrain_epochs', '4',
     '--finetune_epochs', '30',
     '--results_file', 'results_ablation.txt'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join();  t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} — see stderr above ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 5: Display results and plot
import os, re
import numpy as np
import matplotlib.pyplot as plt

# Print results table
for fname in ['results_ablation_smoke.txt', 'results_ablation.txt']:
    if os.path.exists(fname):
        print(f'--- {fname} ---')
        print(open(fname).read())

# Bar chart: zero-shot vs fine-tuned AvgBIO per variant
def parse_results(path):
    """Extract (model_name -> {zero_shot: {nmi,ari,asw,avg_bio}, fine_tuned: ...})."""
    if not os.path.exists(path):
        return {}
    text = open(path).read()
    sections = re.split(r'Zero-shot|Fine-tuned', text)
    data = {}
    phase_keys = ['zero_shot', 'fine_tuned']
    for i, phase in enumerate(phase_keys):
        if i + 1 >= len(sections):
            break
        for line in sections[i + 1].splitlines():
            nums = re.findall(r'\d+\.\d{4}', line)
            if len(nums) == 4:
                name = line[:28].strip()
                if name and not name.startswith(('-', '=', 'M')):
                    if name not in data:
                        data[name] = {}
                    data[name][phase] = {
                        'nmi': float(nums[0]), 'ari': float(nums[1]),
                        'asw': float(nums[2]), 'avg_bio': float(nums[3])
                    }
    return data

data = parse_results('results_ablation.txt')
if data:
    models = list(data.keys())
    metrics = ['nmi', 'ari', 'asw', 'avg_bio']
    metric_labels = ['NMI', 'ARI', 'ASW', 'AvgBIO']
    phases = ['zero_shot', 'fine_tuned']
    phase_labels = ['Zero-shot', 'Fine-tuned']
    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
    bw = 0.35

    for ax, phase, plabel in zip(axes, phases, phase_labels):
        x = np.arange(len(models))
        for mi, (met, mlabel, col) in enumerate(zip(metrics, metric_labels, colors)):
            vals = [data[m].get(phase, {}).get(met, 0.0) for m in models]
            offset = (mi - 1.5) * bw / 4
            bars = ax.bar(x + offset, vals, bw / 4, label=mlabel, color=col, alpha=0.85)
            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x() + bw/8, h + 0.005, f'{h:.3f}',
                        ha='center', va='bottom', fontsize=7, rotation=90)
        ax.set_xticks(x)
        ax.set_xticklabels(models, fontsize=9)
        ax.set_title(f'{plabel} — PBMC-3K Clustering')
        ax.set_ylabel('Score')
        ax.set_ylim(0, 1.15)
        ax.legend(fontsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4)
        ax.set_axisbelow(True)

    fig.suptitle('Cell-JEPA vs Transformer + SIGReg\n2×2 Ablation: Collapse Prevention Mechanism',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig('sigreg_ablation.png', dpi=150)
    plt.show()
    print('Saved sigreg_ablation.png')
else:
    print('No results to plot — did Cell 4 complete?')

In [ ]:
# Cell 6: Save all results to Drive
import shutil, os

RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/sigreg_ablation/'
for f in ['results_ablation.txt', 'results_ablation_smoke.txt', 'sigreg_ablation.png']:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
        print(f'Copied {f}')
    else:
        print(f'Not found: {f} (skipping)')
print(f'Done. Files in {RESULTS_DIR}')